# 🌐 MAKE GRID - NOTEBOOK VERSION

This notebook allows to create a CROCO grid.
The user can evenutally edit the bathymetry and mask.  

See [online documentation](https://croco-ocean.gitlabpages.inria.fr/croco_pytools/prepro/tuto.grid.html) for more details.

## Dependencies
<div style="border-left: 4px solid orange; padding: 10px; background-color: #fff3cd;">
  <strong>⚠️ Warning:</strong> Make sure to build the croco_pyenv environment before using this notebook. 
  
  See https://croco-ocean.gitlabpages.inria.fr/croco_pytools/prepro/tuto.env.html
</div>

In [ ]:
%matplotlib widget  
%load_ext autoreload
%autoreload 2


# Import custom modules
from Modules.croco_class import CROCO

## Directories and inputs and outputs files

<div style="border-left: 4px solid #ccc; background-color: #f8f9fa; padding: 10px; margin: 10px 0;">

To build a CROCO grid two input data are necessary:

- bathymetry data (netcdf), examples available in DATASETS_CROCOTOOLS/Topo, and any other data can be used by editing the readers.jsonc file
- coastline data (shapefile), example available in DATASETS_CROCOTOOLS/gshhs

  The user specifies in section Croco_Files and Grid_Input_Files:

| Variable          | Description                                             |
|-------------------|---------------------------------------------------------|
| `croco_files_dir` | Path of the CROCO files directory                       |
| `croco_grd_prefix`| Prefix for the CROCO grid file that will be created     |
| `topo_file`       | Path/name of the source bathymetry file                 |
| `topo_file_reader`| Keyword for the data reader to use in ``readers.jsonc`` | 
| `shp_file`        | Path/name of the source coastline file                  |

In [ ]:
# Read the config file and create a CROCO instance
croco = CROCO("Examples/benguela_multifiles/grid.ini")

# Edit the appropriate section in the config file
croco.edit_section_widget("Croco_Files")

In [ ]:
# Edit the appropriate section in the config file
croco.edit_section_widget("Grid_Input_Files")

## Grid positionning

<div style="border-left: 4px solid #ccc; background-color: #f8f9fa; padding: 10px; margin: 10px 0;">

  The user can select either a curvilinear grid or a rectangular grid. This choice is controlled by the 
  grid_type parameter. Depending on the selected grid type, different grid parameters must be specified: 
  [central_lon, central_lat, size_x_km, size_y_km, npoints_x, npoints_y, and grid_angle] for the curvilinear gird , 
  [lon_min, lon_max, lat_min, lat_max, dlon, and dlat] for the rectangular grid. 
  The curvilinear grid will be created as an orthogonal grid with minimal gridsize variation. It uses a Mercator 
  projection around the equator and then rotates the sphere around its three axes to position the grid wherever it is located.

  The user specifies: 
| Variable      | Description                                              |
|---------------|----------------------------------------------------------|
| `grid_type`   | Must be either curvilinear or rectangular.               |
|               | Otherwise, an error is raised. If this parameter         |
|               | is not specified, a curvilinear grid is used by default  |
| `central_lon` | Central longitude of the grid                            |
| `central_lat` | Central latitude of the grid                             |
| `size_x_km`   | Size of the domain along x in km                         |
| `size_y_km`   | Size of the domain along y in km                         |
| `npoints_x`   | Number of points along x                                 |
| `npoints_y`   | Number of points along y                                 |
| `grid_angle`  | Rotation angle of the grid (trigonometric convention)    |
| `lon_min`     | Minimum longitude of the rectangular grid                |
| `lon_max`     | Maximum longitude of the rectangular grid                |
| `lat_min`     | Minimum latitude of the rectangular grid                 |
| `lat_max`     | Maximum latitude of the rectangular grid                 |
| `dlon`        | Longitudinal grid spacing of the rectangular grid        |
| `dlat`        | Latitudinal grid spacing of the rectangular grid         |

Note: For the curvilinear grid, the resolution is given by dividing the domain size by the number of point in each direction (size_x_km/npoints_x, in km). 

</div>

In [ ]:
# Edit the appropriate section in the config file
croco.edit_section_widget("Grid_Position")

In [ ]:
# Create lon/lat grid
croco.create_grid()

In [ ]:
# Display the grid
#   with cartopy coastline
croco.plot_grid_outline()
#   with your shapefile coastline
# croco.plot_grid_outline(shp_file=croco.config["Grid_Input_Files"]["shp_file"])
# or
# croco.plot_grid_outline(shp_file="other_shapefile_path")

## Bathymetry interpolation and smoothing and mask creation

<div style="border-left: 4px solid #ccc; background-color: #f8f9fa; padding: 10px; margin: 10px 0;">

  <strong> Bathymetry </strong>

  The bathymetry is interpolated on the CROCO grid using weighted averaging. Weights are computed with a Hann window and more precisely with its polynomial fit (less costly). When input data is coarser than CROCO grid and no data is found in the radius window, a bi-linear interpolation is performed with the four closest points.

  <strong> Mask </strong>

  Mask creation is handled by the python library regionmask and can handle any shapefile to define it. CROCO can manage wetting and drying process in coastal areas, in this case the land/sea boundary in CROCO should be extended further than the usual coastline. In that case, the coastline data will be ignored, and the topography data will be used instead. The minimum depth, hmin, thus needs to be set to a value lower than zero. The mask will be drawn at hmin altitude.
  
  <strong> Removing isolated water bodies </strong>

  In some cases, mask generation can leave water bodies inside the land (lakes for example) which are not connected to the main water body. If not wanted, the isolated water bodies can be masked using the `single_connect` functionnality. It takes a user-specified initial point, which is assumed to be in the main water body and, starting from this point, considers the adjacent non-masked points as “water”. Extending further, non-masked point are also marked as “water” if at least one its neighbouring points is already marked as “water”. At the end, remaining non-connected water bodies, not marked as “water”, because not connected to the main water body, are masked.

| Variable                    | Description |
|-----------------------------|-------------|
| `mask_isolated_waterbodies` | `True/False` to activate/deactivate functionality                   |
| `main_water_body_x_idx`     | Index of the starting point (in the main water body) in x-direction |
| `main_water_body_y_idx`     | Index of the starting point (in the main water body) in y-direction |

  <strong> Smoothing the bathymetry </strong>

 Since CROCO is a sigma-coordinate model, it is sensitive to pressure gradient errors. Although these are accounted for and reduced by the numerical methods used in the model, it is however necessary to smooth the bathymetry so that the slope is everywhere less than a defined r-factor. The default value for r-factor is set to 0.2 but can be changed by the user. The lower the value, the smoother the topography.

  Before smoothing, routine uses hmin and hmax to bound topography before smoothing. Topography values lower than hmin, repectively higher than hmax, will be set to hmin, repectively hmax.

  Criteria specified by the user:

| Variable      | Description |
|---------------|-------------|
| `hmin`        | Minimum depth [m] used at the shore. Depends on resolution (e.g., `dx=100km → hmin=300`, `dx=25km → hmin=150`). Affects smoothing, which works on `grad(h)/h`. |
| `hmax`        | Maximum depth [m]. Limits bathymetry to prevent extrapolation below available input levels. |
| `interp_rad`  | Interpolation radius (in grid points) for the CROCO grid. Typically between 2 and 8. |
| `rfact`       | Maximum slope parameter, `r-fact = grad(h)/h`, used during smoothing. Lower values yield smoother results. |
| `smooth_meth` | Smoothing method to use. Options: `'smooth'`, `'lsmooth'`, `'lsmooth_legacy'`, `'lsmooth2'`, `'lsmooth1'`, `'cond_rx0_topo'`. See [online documentation](https://croco-ocean.gitlabpages.inria.fr/croco_pytools/prepro/tuto.grid.html) for more details. |


 </div>


In [ ]:
# Edit the appropriate section in the config file
croco.edit_section_widget("Grid_Smoothing_Params")

In [ ]:
# Edit the appropriate section in the config file
croco.edit_section_widget("Grid_Isolated_Waterbodies")

In [ ]:
# Interpolate bathymetry, create mask and smooth bathymetry
croco.create_mask_and_topo()

In [ ]:
# Display the bathymetry and mask
croco.plot_h()

## 💾 Save your grid and settings

In [ ]:
# Save CROCO grid in a netcdf file define as croco_grd
croco.save_grid_nc()

# Save your CROCO grid settings for later use
croco.save_config("grid_new.ini")

#
# -- ADVANCED FEATURES --
#

## 🔧 Manually edit the mask or bathymetry

<div style="border-left: 4px solid #ccc; background-color: #f8f9fa; padding: 10px; margin: 10px 0;">

You can directly call the map_editor from your previous croco instance 
or you can load a croco grid file previously generated. 
In this case, uncomment the loading lines.  


In [ ]:
# Load a grid file
# croco = CROCO()  # init without config file
# croco.load_grid("../results/CROCO_FILES/croco_grd.nc")

# Edit the bathymetry
croco.map_editor("h")
# or using your own shapefile for the coastline
# croco.map_editor("h", shp_file=shp_filepath)

In [ ]:
# Edit the mask
croco.map_editor("maskr")

In [ ]:
# Smooth the bathymetry after bathy/mask edition
# Note : if you have loaded from a previous file, you have to load the associated config file
# croco.load_ini("grid.ini")
croco.smooth_after_mask_edit()

In [ ]:
# Display the bathymetry and mask
croco.plot_h()

In [ ]:
# Eventually edit the CROCO file prefix (if you want to keep the original croco_grd.nc unchanged)
croco.edit_section_widget("Croco_Files")

In [ ]:
# Save CROCO grid in a netcdf file define as croco_grd
croco.save_grid_nc()

# Save your CROCO grid settings for later use
croco.save_config("grid_new.ini")